# Exercise 06: k-Nearest Neighbours, Support Vector Machines and Naive Bayes

**Dataset 1 – Palmer penguins:** body measurements of 344 penguins of three species (Adelie, Chinstrap, Gentoo) observed
on three islands in the Palmer Archipelago, Antarctica (Gorman, Williams & Fraser, 2014; Horst, Hill & Gorman, 2020).
Source: [Kaggle – Palmer Archipelago (Antarctica) penguin data](https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data).

| variable | meaning |
|---|---|
| `species` | Adelie, Chinstrap, Gentoo – **target** |
| `culmen_length_mm`, `culmen_depth_mm` | length and depth of the bill (mm) |
| `flipper_length_mm` | flipper length (mm) |
| `body_mass_g` | body mass (g) |
| `island`, `sex` | island and sex |

**Dataset 2 – SMS Spam Collection:** 5'572 real SMS messages in English, labelled as `ham` (legitimate) or `spam`
(Almeida, Gómez Hidalgo & Yamakami, 2011). Source:
[UCI](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) /
[Kaggle](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset).

**Questions:** Can the species of a penguin be determined from a few body measurements? Can spam messages be filtered
automatically based on the words they contain?

**Slides:** Part 04 «Classification models» – k-NN, Naive Bayes, SVM.

**After this exercise you can ...**
- apply k-NN, explain why features must be scaled and choose $k$ by cross-validation,
- explain the role of the margin, the support vectors, $C$ and the kernel of an SVM,
- compute a Naive Bayes classification by hand and apply Naive Bayes to text data.

## Concept

**k-nearest neighbours (k-NN).** To classify a new point $x$: compute the distance to all training points, take the $k$
nearest $\mathcal N_k(x)$ and predict the majority class. Class probability = share of the $k$ neighbours in class $c$.
Euclidean distance:

$$d(x, x') = \sqrt{\sum_{j=1}^{p}(x_j - x'_j)^2}$$

**Caution:** features with large units dominate the distance → standardise first: $z_j = (x_j - \bar x_j)/s_j$.
Small $k$ → flexible (high variance), large $k$ → smooth (high bias).

**Naive Bayes.** Bayes' theorem with the "naive" assumption that the features are independent given the class:

$$P(y \mid x_1,\dots,x_p) \;\propto\; P(y)\,\prod_{j=1}^{p} P(x_j \mid y)$$

Start from the base rate $P(y)$ and multiply one probability per feature; predict the class with the highest score.
Normalise the scores to obtain probabilities. For text, $P(\text{word} \mid y)$ is estimated from word counts
(`MultinomialNB`; with Laplace smoothing so that unseen words do not give a probability of 0).

**Support vector machine (SVM).** Finds the boundary with the **widest margin** between the classes. Only the points on
or inside the margin – the **support vectors** – determine the boundary. $C$ controls the soft margin: small $C$ → wide
margin, some errors allowed; large $C$ → few training errors, narrow margin (risk of overfitting). A **kernel** (e.g. RBF)
measures similarity in a transformed space → curved boundaries. SVMs also need scaled features.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                             classification_report)

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: k-NN and Naive Bayes by hand

**a) k-NN with and without scaling.** A new penguin has a bill length of 45 mm and a body mass of 4'000 g. Five known
penguins:

| penguin | species | bill length [mm] | body mass [g] |
|---|---|---|---|
| P1 | Adelie | 38 | 4000 |
| P2 | Chinstrap | 46 | 3700 |
| P3 | Chinstrap | 48 | 3800 |
| P4 | Adelie | 39 | 3950 |
| P5 | Gentoo | 46 | 5000 |

1. Compute the Euclidean distances on the **original scale** and classify the new penguin with $k=3$.
2. Standardise the differences with the standard deviations of the whole data set (bill length: 5.5 mm, body mass:
   800 g), i.e. use $\Delta z = \Delta x / s$, and classify again with $k=3$.
3. Which result is more plausible? Why?

**b) Naive Bayes for spam.** In a training set of 100 SMS, 20 are spam and 80 are ham. The word "free" occurs in 12 of
the 20 spam and in 4 of the 80 ham messages; the word "call" occurs in 10 spam and 16 ham messages. A new message
contains both "free" and "call". Compute the Naive Bayes scores (as on the slide «Naive Bayes classifier», using only the
two words that occur) and the probability that the message is spam.

**c)** Verify your results with Python.

In [ ]:
# a) k-NN with and without scaling
new = np.array([45, 4000])
X_known = np.array([[38, 4000], [46, 3700], [48, 3800], [39, 3950], [46, 5000]])
species = np.array(['Adelie', 'Chinstrap', 'Chinstrap', 'Adelie', 'Gentoo'])
sd = np.array([5.5, 800])
# Your code here

# b) Naive Bayes
# Your code here

**Your answer:** ...

## Exercise 2: Explore the penguin data

a) Import the penguin data, remove rows with missing values and the row with `sex == '.'`.

b) Create a scatter plot matrix (`sns.pairplot(..., hue='species')`) of the four numerical measurements. Which pairs of
features separate the species well? Compare the scales of the features.

c) Split the data into a stratified training (80 %) and test set (20 %) with `random_state=42`, using the four numerical
measurements as features.

In [ ]:
# a) Import the data
peng = pd.read_csv('./Data/penguins.csv', sep=',')
# Your code here

# b) Pairplot
features = ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']
# Your code here

# c) Train/test split
X_train, X_test, y_train, y_test = ___

**Your answer:** ...

## Exercise 3: k-NN – the effect of scaling

a) Compute the 5-fold CV accuracy (on the training set) of `KNeighborsClassifier(n_neighbors=5)` on the **unscaled**
features and of `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))`.

b) Explain the difference. Why must the scaler be part of the pipeline (and not be fitted on the whole data set
beforehand)?

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Your code here

**Your answer:** ...

## Exercise 4: k-NN – choosing k

a) Compute the 5-fold CV accuracy of the scaled k-NN for $k = 1, 2, \dots, 40$ and plot it against $k$.

b) Choose $k$ (if several values are equally good, prefer the larger one – it gives a smoother model). Fit the model on
the whole training set and compute accuracy and confusion matrix on the test set.

c) Use the model to classify a new penguin with bill length 45 mm, bill depth 18 mm, flipper length 195 mm and body mass
4'000 g. Print the class probabilities (`predict_proba`).

In [ ]:
# a) CV accuracy for k = 1..40
ks = range(1, 41)
# Your code here

# b) Final model and test evaluation
# Your code here

# c) Classify a new penguin
new_penguin = pd.DataFrame({'culmen_length_mm': [45], 'culmen_depth_mm': [18],
                            'flipper_length_mm': [195], 'body_mass_g': [4000]})
# Your code here

**Your answer:** ...

## Exercise 5: Support vector machines – margin, C and kernel

To visualise the decision boundaries we use only the two features `culmen_length_mm` and `culmen_depth_mm`.

a) Fit `make_pipeline(StandardScaler(), SVC(kernel=..., C=...))` for `kernel` ∈ {`'linear'`, `'rbf'`} and
$C$ ∈ {0.01, 1, 100} on the training set. Plot the 6 decision boundaries in a 2 × 3 grid
(the decision regions are drawn by predicting the species on a fine grid), with the training points and the **support vectors** circled.

b) Print the number of support vectors and the 5-fold CV accuracy for each model.

c) Describe the influence of $C$ and of the kernel. Which model would you choose?

In [ ]:
features_2d = ['culmen_length_mm', 'culmen_depth_mm']
X2_train = X_train[features_2d]
colors = {'Adelie': 'tab:orange', 'Chinstrap': 'tab:purple', 'Gentoo': 'tab:green'}

# Grid of points covering the feature space (used to draw the decision regions)
xx, yy = np.meshgrid(np.linspace(32, 61, 300), np.linspace(12.5, 22.5, 300))
grid = pd.DataFrame({features_2d[0]: xx.ravel(), features_2d[1]: yy.ravel()})

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
for row, kernel in enumerate(['linear', 'rbf']):
    for col, C in enumerate([0.01, 1, 100]):
        ax = axes[row, col]
        svm = ___

        # Decision regions: predict the species for every grid point
        zz = pd.Categorical(svm.predict(grid), categories=list(colors)).codes.reshape(xx.shape)
        ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5, 2.5], colors=list(colors.values()), alpha=0.2)

        # Plot the training points (coloured by species) and circle the support vectors
        # (support vector indices: svm[-1].support_)
        # Your code here
plt.show()

**Your answer:** ...

## Exercise 6: Naive Bayes spam filter

a) Import the SMS data (`encoding='latin-1'`), keep only the first two columns and rename them to `label` and `text`.
How many messages are spam? Remove duplicated messages.

b) Split the data into a stratified training (80 %) and test set (20 %) with `random_state=42`.

c) Transform the texts into word counts with `CountVectorizer(stop_words='english')` (fit on the training texts only!)
and fit a `MultinomialNB()`. How many different words (features) are there?

d) Evaluate the model on the test set (confusion matrix, classification report). Which error is worse for a spam filter:
a false positive (ham classified as spam) or a false negative?

In [ ]:
# a) Import the data
sms = pd.read_csv('./Data/sms_spam.csv', encoding='latin-1')
# Your code here

# b) Train/test split
# Your code here

# c) Bag of words and Naive Bayes
vectorizer = CountVectorizer(stop_words='english')
# Your code here

# d) Evaluation
# Your code here

**Your answer:** ...

## Exercise 7: What has the spam filter learned?

a) The attribute `nb.feature_log_prob_` contains $\log P(\text{word} \mid \text{class})$ for both classes. Compute for
each word the log ratio $\log P(\text{word} \mid \text{spam}) - \log P(\text{word} \mid \text{ham})$ and print the 15
words that indicate spam most strongly.

b) Classify the following messages and print $P(\text{spam})$. Write two messages of your own.

In [ ]:
# a) Most indicative spam words
words = vectorizer.get_feature_names_out()
# Your code here

# b) Classify new messages
new_messages = ['Congratulations! You have won a free prize. Call now to claim your reward',
                'Hi, are we still meeting for lunch tomorrow?',
                'URGENT: your account has been selected, reply YES to claim cash',
                'Can you call me when you are free?']
# Your code here

**Your answer:** ...

## Exercise 8: Reflection

Compare k-NN, SVM and Naive Bayes: for which of the three methods is

a) feature scaling necessary,

b) the training fast but the prediction slow,

c) a very large number of features (e.g. thousands of words) unproblematic?

**Your answer:** ...

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')